# 🪙 Entrenar RF-DETR para contar monedas argentinas (gratis en Google Colab)

Este cuaderno toma el **dataset que exportaste desde la web** (botón *Exportar* en la pestaña *Etiquetar*), entrena un modelo **RF-DETR Nano** y genera los dos archivos que vas a cargar en la web del celular:

- `monedas_rfdetr.onnx` (el modelo)
- `classes.json` (nombres de clases, resolución y calibración)

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* (gratis). Después ejecutá las celdas en orden (▶).

In [ ]:
#@title 1 · Instalar dependencias
!nvidia-smi -L || echo "⚠️ No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"
!pip -q install "rfdetr[onnx]" onnxruntime onnxconverter-common

In [ ]:
#@title 2 · Subir el dataset (.zip exportado desde la web)
import os, zipfile, glob, json, shutil
from collections import Counter, defaultdict
from google.colab import files

DATASET_DIR = "/content/dataset"
subido = files.upload()                      # elegí dataset_monedas_AAAA-MM-DD.zip
zip_path = next(iter(subido))
shutil.rmtree(DATASET_DIR, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATASET_DIR)

# por si el zip tiene una carpeta raíz
if not os.path.exists(f"{DATASET_DIR}/train/_annotations.coco.json"):
    cand = glob.glob(f"{DATASET_DIR}/**/train/_annotations.coco.json", recursive=True)
    assert cand, "No encontré train/_annotations.coco.json dentro del zip"
    DATASET_DIR = os.path.dirname(os.path.dirname(cand[0]))

for split in ["train", "valid", "test"]:
    d = json.load(open(f"{DATASET_DIR}/{split}/_annotations.coco.json"))
    cats = {c["id"]: c["name"] for c in d["categories"]}
    cnt = Counter(cats[a["category_id"]] for a in d["annotations"])
    print(f"{split:5s}: {len(d['images']):4d} imágenes ·", dict(cnt))

### ¿Varias tandas de fotos?
Si etiquetaste en varios teléfonos o días, podés exportar varios zip y entrenar con el último que contenga todo (las fotos quedan guardadas en la web hasta que las borres).

In [ ]:
#@title 3 · Entrenar
from rfdetr import RFDETRNano     # Nano = el más rápido en el celular. Si te falta precisión probá RFDETRSmall (512 px)

MODELO     = RFDETRNano
OUT_DIR    = "/content/salida"
EPOCHS     = 60                    # 40–100 suele andar bien; más fotos → podés subirlo
BATCH      = 8                     # si da error de memoria (CUDA OOM) bajalo a 4 y subí GRAD_ACCUM a 4
GRAD_ACCUM = 2

model = MODELO()
model.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=BATCH,
    grad_accum_steps=GRAD_ACCUM,
    lr=1e-4,
    output_dir=OUT_DIR,
)

In [ ]:
#@title 4 · Exportar a ONNX
EXPORT_DIR = "/content/export"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)

preferidos = ["checkpoint_best_total.pth", "checkpoint_best_ema.pth", "checkpoint_best_regular.pth", "checkpoint.pth"]
ckpts = [f"{OUT_DIR}/{p}" for p in preferidos if os.path.exists(f"{OUT_DIR}/{p}")]
print("Checkpoints:", ckpts)
try:
    best = MODELO(pretrain_weights=ckpts[0])
    best.export(output_dir=EXPORT_DIR)
except Exception as e:
    print("No pude recargar el mejor checkpoint (", e, ") → exporto el modelo recién entrenado")
    model.export(output_dir=EXPORT_DIR)

onnx_files = [p for p in glob.glob(f"{EXPORT_DIR}/**/*.onnx", recursive=True) if "backbone" not in p]
ONNX_PATH = onnx_files[0]
print("ONNX:", ONNX_PATH, f"{os.path.getsize(ONNX_PATH)/1e6:.1f} MB")

In [ ]:
#@title 5 · Verificar el ONNX y generar classes.json (calibra el índice de clases)
import numpy as np, onnxruntime as ort
from PIL import Image

sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
RES = int(inp.shape[-1])
OUT_NAMES = [o.name for o in sess.get_outputs()]
print("Entrada:", inp.name, inp.shape, "· Salidas:", OUT_NAMES)
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

def correr(session, path):
    # mismo preprocesado que la web: resize cuadrado + normalización ImageNet
    im = Image.open(path).convert("RGB"); W, H = im.size
    x = (np.asarray(im.resize((RES, RES), Image.BILINEAR), np.float32) / 255 - MEAN) / STD
    o = dict(zip(OUT_NAMES, session.run(None, {inp.name: x.transpose(2, 0, 1)[None]})))
    boxes = o.get("dets", list(o.values())[0])[0]; logits = o.get("labels", list(o.values())[1])[0]
    if boxes.shape[-1] != 4: boxes, logits = logits, boxes
    cx, cy, w, h = boxes.T
    xyxy = np.stack([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H], 1)
    return xyxy, 1 / (1 + np.exp(-logits))

def iou(g, b):
    ix = np.clip(np.minimum(g[2], b[:, 2]) - np.maximum(g[0], b[:, 0]), 0, None)
    iy = np.clip(np.minimum(g[3], b[:, 3]) - np.maximum(g[1], b[:, 1]), 0, None)
    I = ix * iy
    return I / ((g[2]-g[0]) * (g[3]-g[1]) + (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1]) - I + 1e-9)

def recolectar(split, max_imgs=200):
    ann = json.load(open(f"{DATASET_DIR}/{split}/_annotations.coco.json"))
    by_img = defaultdict(list)
    for a in ann["annotations"]: by_img[a["image_id"]].append(a)
    res = []
    for im in ann["images"][:max_imgs]:
        xyxy, probs = correr(sess, f"{DATASET_DIR}/{split}/{im['file_name']}")
        for a in by_img[im["id"]]:
            x, y, w, h = a["bbox"]; g = np.array([x, y, x + w, y + h])
            ious = iou(g, xyxy); q = int(ious.argmax())
            res.append((a["category_id"], ious[q], probs[q]))
    return ann, res

ann, pares = recolectar("valid")
if len(pares) < 10:
    ann, extra = recolectar("train"); pares += extra

cats = sorted([c for c in ann["categories"] if c.get("supercategory") != "none"], key=lambda c: c["id"])
NAMES = [c["name"] for c in cats]
id2idx = {c["id"]: i for i, c in enumerate(cats)}

# offset = índice de logit de la clase 0 (RF-DETR puede conservar el id 0 de la supercategoría)
dif = Counter(int(p.argmax()) - id2idx[cid] for cid, i, p in pares if i >= 0.5)
OFFSET = dif.most_common(1)[0][0] if dif else 1
print("Offset de clases detectado:", OFFSET, dict(dif))

# precisión por clase sobre las cajas que el modelo encontró
ok, tot, encontradas = Counter(), Counter(), 0
for cid, i, p in pares:
    tot[cid] += 1
    if i >= 0.5:
        encontradas += 1
        k = int(p[OFFSET:OFFSET + len(NAMES)].argmax())
        ok[cid] += int(k == id2idx[cid])
print(f"\nMonedas encontradas (IoU≥0.5): {encontradas}/{len(pares)}")
for c in cats:
    t = tot[c["id"]]
    pct = f"{100 * ok[c['id']] / t:5.1f}%" if t else "sin datos"
    print(f"  {c['name']:14s} {ok[c['id']]:4d}/{t:<4d} {pct}")

meta = {"names": NAMES, "offset": OFFSET, "resolution": RES, "file": "monedas_rfdetr.onnx",
        "modelo": MODELO.__name__}
os.makedirs("/content/para_la_web", exist_ok=True)
json.dump(meta, open("/content/para_la_web/classes.json", "w"), indent=2, ensure_ascii=False)
print("\nclasses.json:", meta)

In [ ]:
#@title 6 · (Recomendado) Versión FP16: la mitad de tamaño, igual de precisa en la práctica
USAR_FP16 = True
final_path = "/content/para_la_web/monedas_rfdetr.onnx"
shutil.copy(ONNX_PATH, final_path)
if USAR_FP16:
    try:
        import onnx
        from onnxconverter_common import float16
        m16 = float16.convert_float_to_float16(onnx.load(ONNX_PATH), keep_io_types=True)
        p16 = "/content/export/modelo_fp16.onnx"; onnx.save(m16, p16)
        s16 = ort.InferenceSession(p16, providers=["CPUExecutionProvider"])
        img0 = glob.glob(f"{DATASET_DIR}/valid/*.jpg")[0]
        a = correr(sess, img0)[1]; b = correr(s16, img0)[1]
        diff = float(np.abs(a - b).max()); print("Diferencia máx. de probabilidad fp32 vs fp16:", round(diff, 4))
        if diff < 0.05:
            shutil.copy(p16, final_path); print("✓ Uso la versión FP16")
        else:
            print("FP16 difiere demasiado → me quedo con FP32")
    except Exception as e:
        print("FP16 no disponible (", e, ") → uso FP32")
print(f"Modelo final: {os.path.getsize(final_path)/1e6:.1f} MB")

In [ ]:
#@title 7 · Descargar los archivos para la web
files.download("/content/para_la_web/monedas_rfdetr.onnx")
files.download("/content/para_la_web/classes.json")
# Alternativa: guardarlos en Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copytree('/content/para_la_web', '/content/drive/MyDrive/monedas_modelo', dirs_exist_ok=True)

### Siguiente paso
En el celular: **Ajustes → Cargar modelo** y elegí juntos `monedas_rfdetr.onnx` y `classes.json`. Queda guardado en el teléfono.

**Ciclo de mejora:** cargá el modelo → en *Etiquetar* usá **Pre-etiquetar** para que proponga cajas en fotos nuevas → corregí → exportá → re-entrená. Cada vuelta es más rápida.